In [ ]:
import pandas as pd
from sklearn.metrics import f1_score
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier

---
# Modèle Random Forest Classifier

---
## Validation croisée

In [3]:
df_x_clean = pd.read_csv('../data/x_train_clean.csv')
df_y_clean = pd.read_csv('../data/y_train_clean.csv')
df_x_test_clean = pd.read_csv('../data/x_test_clean.csv')


# les 32 colonnes d'entreprise
company_cols = [f'company_feature_{i}' for i in range(32)]

#regroupe sous un même identifiant les lignes qui ont la meme entreprise
df_x_clean['company_group'] = df_x_clean[company_cols].apply(lambda row: hash(tuple(row)), axis=1)
groups_company = df_x_clean['company_group']


#on va faire une validation croisée en utilisant StratifiedGroupKFold
#le découpage respecte à la fois les groupes d'entités et l'équilibre des classes dans chaque bloc.

X = df_x_clean.drop(columns=['id', 'company_group'])
y = df_y_clean['target']

cv = StratifiedGroupKFold(n_splits=5,random_state=42, shuffle=True)
scores_f1 = []

#entraînement et d'évaluation pli par pli
for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups=groups_company)):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    
    model = RandomForestClassifier(
        class_weight='balanced',
        random_state=42,
    )
    
    model.fit(X_tr, y_tr)
    
    preds = model.predict(X_te)
    score = f1_score(y_te, preds, average='macro')
    scores_f1.append(score)
    
    print(f"Pli {fold + 1} - F1-Macro : {score}")

print(f"\n---> F1-Macro Moyen : {np.mean(scores_f1)}")


Pli 1 - F1-Macro : 0.30922390308147624
Pli 2 - F1-Macro : 0.30959726456437586
Pli 3 - F1-Macro : 0.27533464979325756
Pli 4 - F1-Macro : 0.32191006822848905
Pli 5 - F1-Macro : 0.27179397088699664

---> F1-Macro Moyen : 0.2975719713109191


In [4]:
#entraînement de ce modèle sur tout le jeu de données d'entraînement
df_x_clean = pd.read_csv('../data/x_train_clean.csv')
df_y_clean = pd.read_csv('../data/y_train_clean.csv')
df_x_test_clean = pd.read_csv('../data/x_test_clean.csv')

X_train = df_x_clean.drop(columns=['id'])
y_train = df_y_clean['target']
X_test = df_x_test_clean.drop(columns=['id'])

model_final1 = RandomForestClassifier(
    class_weight='balanced',
    random_state=42
)

model_final1.fit(X_train, y_train)

# prédiction sur le jeu de test
predictions = model_final1.predict(X_test)

# repasser des nombres aux noms de postes
postes = {
    0: "Assistant",
    1: "Executive",
    2: "Manager",
    3: "Director"
}
predictions_labels = [postes[p] for p in predictions]

# créer le fichier
predictions_final_1 = pd.DataFrame({
    'id': df_x_test_clean['id'],
    'position': predictions_labels
})
predictions_final_1.to_csv('../results/predictions_RandomForest_1.csv', index=True)


**Le score de ces prédictions est 0,2945277329615221**

---
## Recherche hyperparamètres

In [ ]:
df_x_clean = pd.read_csv('../data/x_train_clean.csv')
df_y_clean = pd.read_csv('../data/y_train_clean.csv')
df_x_test_clean = pd.read_csv('../data/x_test_clean.csv')

# les 32 colonnes d'entreprise
company_cols = [f'company_feature_{i}' for i in range(32)]

#regroupe sous un même identifiant les lignes qui ont la meme entreprise
df_x_clean['company_group'] = df_x_clean[company_cols].apply(lambda row: hash(tuple(row)), axis=1)
groups_company = df_x_clean['company_group']

#on va faire une validation croisée en utilisant StratifiedGroupKFold
#le découpage respecte à la fois les groupes d'entités et l'équilibre des classes dans chaque bloc.

X = df_x_clean.drop(columns=['id', 'company_group'])
y = df_y_clean['target']


# 2. hyperparamètres à tester
param_test = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [8, 12, 16, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', 0.5, 0.8]
}


model1 = RandomForestClassifier(
    class_weight='balanced',
    random_state=42
)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# recherche aléatoire hyperparamètres
recherche = RandomizedSearchCV(
    estimator=model1,
    param_distributions=param_test,
    n_iter=20,
    scoring='f1_macro',
    cv=cv,
    random_state=42,
    n_jobs=-1
)
recherche.fit(X, y, groups=groups_company)


print(f"Meilleur F1-Macro: {recherche.best_score_:.4f} ---")
print("\nMeilleurs hyperparamètres trouvés :")
print(recherche.best_params_)

--- Meilleur F1-Macro Réel (GroupKFold) : 0.3487 ---

Meilleurs hyperparamètres trouvés :
{'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 20}


On va maintenant entrainer le même modèle avec les meilleurs hyperparamètres trouvés sur l'ensemble des données train 

Meilleurs hyperparamètres:  
'n_estimators': 100  
'min_samples_split': 5  
'min_samples_leaf': 4  
'max_features': 'sqrt'  
'max_depth': 20  


In [ ]:
#entraînement du modéle avec les meilleurs hyperparamètres sur tout le jeu de données d'entraînement
df_x_clean = pd.read_csv('../data/x_train_clean.csv')
df_y_clean = pd.read_csv('../data/y_train_clean.csv')
df_x_test_clean = pd.read_csv('../data/x_test_clean.csv')

X_train = df_x_clean.drop(columns=['id'])
y_train = df_y_clean['target']
X_test = df_x_test_clean.drop(columns=['id'])

model_final2 = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_estimators=recherche.best_params_['n_estimators'],
    max_depth=recherche.best_params_['max_depth'],
    min_samples_split=recherche.best_params_['min_samples_split'],
    min_samples_leaf=recherche.best_params_['min_samples_leaf'],
    max_features=recherche.best_params_['max_features']
)

model_final2.fit(X_train, y_train)

# prédiction sur le jeu de test
predictions = model_final2.predict(X_test)

# repasser des nombres aux noms de postes
postes = {
    0: "Assistant",
    1: "Executive",
    2: "Manager",
    3: "Director"
}
predictions_labels = [postes[p] for p in predictions]

# créer le fichier
predictions_final_2 = pd.DataFrame({
    'id': df_x_test_clean['id'],
    'position': predictions_labels
})
predictions_final_2.to_csv('../results/predictions_RandomForest_2.csv', index=True)

**Le score de ces predictions est 0,32167326229874066**

## Nouvelle optimisation des hyperparamètres

In [ ]:
df_x_clean = pd.read_csv('../data/x_train_clean.csv')
df_y_clean = pd.read_csv('../data/y_train_clean.csv')
df_x_test_clean = pd.read_csv('../data/x_test_clean.csv')

# les 32 colonnes d'entreprise
company_cols = [f'company_feature_{i}' for i in range(32)]

#regroupe sous un même identifiant les lignes qui ont la meme entreprise
df_x_clean['company_group'] = df_x_clean[company_cols].apply(lambda row: hash(tuple(row)), axis=1)
groups_company = df_x_clean['company_group']

#on va faire une validation croisée en utilisant StratifiedGroupKFold
#le découpage respecte à la fois les groupes d'entités et l'équilibre des classes dans chaque bloc.

X = df_x_clean.drop(columns=['id', 'company_group'])
y = df_y_clean['target']


# 2. hyperparamètres à tester
param_test = {
    'n_estimators': [50, 75, 100, 200, 300],
    'max_depth': [16, 20, 30, 40],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', 0.5, 0.8]
}


model1 = RandomForestClassifier(
    class_weight='balanced',
    random_state=42
)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# recherche aléatoire hyperparamètres
recherche = RandomizedSearchCV(
    estimator=model1,
    param_distributions=param_test,
    n_iter=20,
    scoring='f1_macro',
    cv=cv,
    random_state=42,
    n_jobs=-1
)
recherche.fit(X, y, groups=groups_company)


print(f"--- Meilleur F1-Macro Réel (GroupKFold) : {recherche.best_score_:.4f} ---")
print("\nMeilleurs hyperparamètres trouvés :")
print(recherche.best_params_)

--- Meilleur F1-Macro Réel (GroupKFold) : 0.3429 ---

Meilleurs hyperparamètres trouvés :
{'n_estimators': 75, 'min_samples_split': 10, 'min_samples_leaf': 8, 'max_features': 'log2', 'max_depth': 40}


In [ ]:
#entraînement du modèle avec les meilleurs hyperparamètres sur tout le jeu de données d'entraînement
df_x_clean = pd.read_csv('../data/x_train_clean.csv')
df_y_clean = pd.read_csv('../data/y_train_clean.csv')
df_x_test_clean = pd.read_csv('../data/x_test_clean.csv')

X_train = df_x_clean.drop(columns=['id'])
y_train = df_y_clean['target']
X_test = df_x_test_clean.drop(columns=['id'])

model_final3 = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_estimators=recherche.best_params_['n_estimators'],
    max_depth=recherche.best_params_['max_depth'],
    min_samples_split=recherche.best_params_['min_samples_split'],
    min_samples_leaf=recherche.best_params_['min_samples_leaf'],
    max_features=recherche.best_params_['max_features']
)

model_final3.fit(X_train, y_train)

# prédiction sur le jeu de test
predictions = model_final3.predict(X_test)

# repasser des nombres aux noms de postes
postes = {
    0: "Assistant",
    1: "Executive",
    2: "Manager",
    3: "Director"
}
predictions_labels = [postes[p] for p in predictions]

# créer le fichier
predictions_final_3 = pd.DataFrame({
    'id': df_x_test_clean['id'],
    'position': predictions_labels
})
predictions_final_3.to_csv('../results/predictions_RandomForest_3.csv', index=True)